In [4]:
from collections import defaultdict


class GrafoDAG:
    def __init__(self, vertices):
      self.V = vertices
      self.adyacencia = defaultdict(list)

    def agregar_arista(self, u,v,peso):
        self.adyacencia[u].append((v,peso))

    def _orden_topologico(self):
        visitado = set()
        pila = []

        def dfs(nodo):
            visitado.add(nodo)
            for vecino, _ in self.adyacencia[nodo]: # Fixed: added space between _ and in
                if vecino not in visitado:
                    dfs(vecino)
            pila.append(nodo)

        for v in self.V:
            if v not in visitado:
                dfs(v)

        return pila[::-1] # Fixed: Reverse the topological order




    def camino_mas_corto(self, fuente): # Moved into class and added self
        INF = float('inf')

        orden = self._orden_topologico()
        dist = {v: INF for v in self.V}
        previo = {v: None for v in self.V}
        dist[fuente] = 0

        print(f"\n orden topologico {' -> '.join(orden)}") # Fixed formatting
        print(f"\n fuente: {fuente} | dist inicial = inf") # Fixed formatting
        print(f"")


        for u in orden:
            if dist[u] == INF: # Fixed: missing colon
                continue

            for v, peso in self.adyacencia[u]: # Fixed: U to u
                nueva_dist = dist[u] + peso # Fixed: U to u
                if nueva_dist < dist[v]:
                    dist[v] = nueva_dist
                    previo[v] = u
                    print(f"{u} {v}  w= {peso} ")
                else:
                    print(f"{u} {v} w={peso} ")

        return dist, previo



    def reconstruir_ruta(self, previo, destino): # Moved into class and added self
        ruta = []
        actual = destino
        while actual is not None: # Fixed: missing colon
            ruta.append(actual)
            actual = previo[actual]
        return ruta[::-1]



    def mostrar_resultados(self, fuente, dist, previo): # Moved into class and added self
        INF = float('inf')
        print(f"\n {'='*46}")
        print(f"resultados desde {fuente}")
        print(f"{'='*46}") # Fixed: f-string syntax
        print(f"{'destino':<10} {'distancia':<12} {'ruta'}") # Fixed: f-string formatting
        print("-"*46)
        for v in self.V:
            if dist[v] == INF:
                print(f"{v:<10} {'INF':>12}") # Fixed: missing value for formatting
            else:
                ruta = self.reconstruir_ruta(previo, v)
                print(f"{v:<10}{dist[v]:<12}{' -> '.join(map(str, ruta))}") # Fixed: f-string syntax and join method



if __name__ == "__main__":

  vertices = ['A','B','C','D','E','F','G']
  g = GrafoDAG(vertices) # Fixed: grafoDAG to GrafoDAG (class name)

  g.agregar_arista('A', 'B', 3)
  g.agregar_arista('A', 'C', 6)
  g.agregar_arista('B', 'D', 2)
  g.agregar_arista('B', 'E', 7)
  g.agregar_arista('C', 'E', 1)
  g.agregar_arista('D', 'F', 5)
  g.agregar_arista('E', 'F', 2)
  g.agregar_arista('E', 'G', 8)
  g.agregar_arista('F', 'G', 1)

dist, previo = g.camino_mas_corto('A')
g.mostrar_resultados('A', dist, previo)



 orden topologico A -> C -> B -> E -> D -> F -> G

 fuente: A | dist inicial = inf

A B  w= 3 
A C  w= 6 
C E  w= 1 
B D  w= 2 
B E w=7 
E F  w= 2 
E G  w= 8 
D F w=5 
F G  w= 1 

resultados desde A
destino    distancia    ruta
----------------------------------------------
A         0           A
B         3           A -> B
C         6           A -> C
D         5           A -> B -> D
E         7           A -> C -> E
F         9           A -> C -> E -> F
G         10          A -> C -> E -> F -> G


In [5]:
import bisect


def list_dp(arr):
    n = len(arr)
    if n== 0:
        return 0, []

    dp = [1]*n
    previo = [-1]*n

    print(f"\nArreglo: {arr}") # Fixed f-string typo 'nArreglo
    print(f"\n{'i':<5} {'arr[i]':<10} {'dp[i]':<8}{'Extiende a'}") # Fixed f-string typo arr[i]:<10'
    print("-"*40)


    for i in range(n):
        for j in range(i):
            if arr[j] < arr[i] and dp[j] + 1 > dp[i]:
                dp[i] = dp[j] + 1
                previo[i] = j
        extiende = f"indice {previo[i]} (valor {arr[previo[i]]})" if previo[i] != -1 else "-"
        print(f"{i:<5} {arr[i]:<10} {dp[i]:<8} {extiende}")

    idx_max = max(range(n), key=lambda i:dp[i])
    subsequencia = []
    idx = idx_max
    while idx != -1:
        subsequencia.append(arr[idx])
        idx = previo[idx]
    subsequencia.reverse()


    return dp[idx_max], subsequencia



def list_binaria(arr):
    n = len(arr)
    if n == 0:
        return 0, []

    colas  = []
    padre = [-1]*n
    pos_en_colas = [-1]*n # Fixed: Initialize pos_en_colas

    print(f"\nArreglo: {arr}")
    print(f"\n{'i':<5}{'arr[i]':<10}{'accion':<30}{'colas'}")
    print("-"*60)

    for i in range(n):
        p = bisect.bisect_left(colas,arr[i])

        if p == len(colas):
            accion = f"Extender (nueva longitud {p+1})"
            colas.append(arr[i])
        else:
            accion = f"Reemplazar colas[{p}] = {colas[p]} por {arr[i]}" # Fixed: Clarified replacement message
            colas[p] = arr[i]


        pos_en_colas[i] = p
        print(f"{i:<5} {arr[i]:<10} {accion:<30} {colas}") # Fixed: i<5 to i:<5

    # Moved subsequence reconstruction and return outside the loop
    longitud = len(colas)
    subsequencia = []
    buscar = longitud - 1
    for i in range(n -1, -1, -1):
        if pos_en_colas[i] == buscar: # Fixed: Check pos_en_colas for reconstruction
            subsequencia.append(arr[i])
            buscar -= 1
            if buscar < 0: # Fixed: missing colon
                break
    subsequencia.reverse()

    return longitud, subsequencia


def comparar(arr):
    print("-"*60)
    print("ENFOQUE 1: programacion dinamica n^2")
    print("-" * 60)
    long_dp, sub_dp = list_dp(arr) # Fixed: lis_dp to list_dp
    print(f"\n longitud {long_dp}")
    print(f"subsequencia: {sub_dp}")


    print("\n" + "=" * 60)
    print("Busqueda binaria n log n")
    print("=" * 60)
    long_bin, sub_bin = list_binaria(arr) # Fixed: lis_binaria to list_binaria
    print(f"\n {long_bin}")
    print(f"{sub_bin}")


    print("\n" + "="*60)
    print(f"arreglo original {arr}")
    print(f"LIS DP {sub_dp} {long_dp}")
    print(f"LIS Binaria {sub_bin} {long_bin}") # Fixed: Added label
    print("="*60)



if __name__ == "__main__": # Fixed: missing colon
    ejemplos = [
        [10, 9, 2, 5, 3, 7, 101, 18],
        [0, 1, 0, 3, 2, 3],
        [3, 10, 2, 1, 20],
    ]


    for arr in ejemplos:
        comparar(arr)
        print()


------------------------------------------------------------
ENFOQUE 1: programacion dinamica n^2
------------------------------------------------------------

Arreglo: [10, 9, 2, 5, 3, 7, 101, 18]

i     arr[i]     dp[i]   Extiende a
----------------------------------------
0     10         1        -
1     9          1        -
2     2          1        -
3     5          2        indice 2 (valor 2)
4     3          2        indice 2 (valor 2)
5     7          3        indice 3 (valor 5)
6     101        4        indice 5 (valor 7)
7     18         4        indice 5 (valor 7)

 longitud 4
subsequencia: [2, 5, 7, 101]

Busqueda binaria n log n

Arreglo: [10, 9, 2, 5, 3, 7, 101, 18]

i    arr[i]    accion                        colas
------------------------------------------------------------
0     10         Extender (nueva longitud 1)    [10]
1     9          Reemplazar colas[0] = 10 por 9 [9]
2     2          Reemplazar colas[0] = 9 por 2  [2]
3     5          Extender (nueva longi

In [8]:
def distancia_edicion(s1, s2):
    m, n = len(s1), len(s2)

    dp = [[0] * (n+1) for _ in range(m+1)] # Fixed: rannge to range

    for i in range(m+1): # Fixed: 'in' to 'i'
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j


    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]

            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],
                    dp[i][j-1],
                    dp[i-1][j-1],
                )

    return dp



def distancia_edicion_optima(s1,s2): # Kept only one definition
    m,n = len(s1), len(s2)
    fila_anterior = list(range(n + 1))

    for i in range(1, m + 1):
        fila_actual = [i] + [0] * n
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                fila_actual[j] = fila_anterior[j - 1]

            else:
                fila_actual[j] = 1 + min(
                    fila_anterior[j],
                    fila_actual[j - 1],
                    fila_anterior[j - 1],
                )
        fila_anterior = fila_actual # Fixed: Indentation, moved out of inner loop

    return fila_anterior[n] # Fixed: Indentation



def reconstruir_operaciones(s1, s2, dp):
    operaciones = []
    i, j = len(s1), len(s2)

    while i > 0 or j > 0:
        if i > 0 and j > 0 and s1[i -1] == s2[j - 1]:
            operaciones.append(('COPIAR', f'{s1[i-1]} en {i-1}')) # Added 'COPIAR' operation for matched characters
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i -1][j - 1] + 1: # Reemplazar
            operaciones.append(('REEMPLAZAR', f'{s1[i-1]} por {s2[j-1]} en {i - 1}')) # Fixed: append arguments, s2[j1] to s2[j-1]
            i -= 1
            j -= 1
        elif j > 0 and dp[i][j] == dp[i][j - 1] + 1: # Insertar
            operaciones.append(('INSERTAR', f'{s2[j-1]} en {i}')) # Fixed: append arguments
            j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1: # Eliminar
            operaciones.append(('ELIMINAR', f'{s1[i - 1]} en {i - 1}')) # Fixed: append arguments
            i -= 1
        else: # Case for i=0 or j=0 when one string is empty (handled by initial dp values)
            # This else block should ideally not be reached if operations are correctly prioritized
            # or if the loop conditions handle boundary cases well. It's usually for deletions/insertions
            # when one string is exhausted.
            break # Exit if no matching operation is found (shouldn't happen with correct logic)

    operaciones.reverse()
    return operaciones



def imprimir_tabla(s1, s2, dp):
    """Muestra la tabla DP con encabezados."""
    m, n = len(s1), len(s2)
    ancho = 4

    # Encabezado de columnas
    encabezado = " " * (ancho * 2) + "".join(f"{'ε':>{ancho}}") + \
                 "".join(f"{c:>{ancho}}" for c in s2)
    print(encabezado)

    for i in range(m + 1):
        etiqueta = "ε" if i == 0 else s1[i - 1]
        fila = f"{etiqueta:>{ancho}} " + "".join(f"{dp[i][j]:>{ancho}}" for j in range(n + 1))
        print(fila)


def imprimir_operaciones(operaciones):
    """Muestra las operaciones en formato tabla."""
    if not operaciones:
        print("  (ninguna operación — las cadenas son iguales)")
        return
    print(f"\n  {'#':<4} {'Operación':<12} {'Detalle'}")
    print("  " + "-" * 48)
    for k, (op, detalle) in enumerate(operaciones, 1):
        simbolo = {"REEMPLAZAR": "↔", "INSERTAR": "+", "ELIMINAR": "−", "COPIAR": "="}[op] # Added 'COPIAR'
        print(f"  {k:<4} {simbolo} {op:<11} {detalle}")


def mostrar_resultado(s1, s2):
    print("\n" + "=" * 56)
    print(f"  s1 = \"{s1}\"")
    print(f"  s2 = \"{s2}\"")
    print("=" * 56)

    dp = distancia_edicion(s1, s2)

    print("\nTabla DP  (dp[i][j] = ediciones para s1[:i] → s2[:j])\n")
    imprimir_tabla(s1, s2, dp)

    distancia = dp[len(s1)][len(s2)]
    print(f"\nDistancia de edición: {distancia}")

    ops = reconstruir_operaciones(s1, s2, dp)
    print(f"\nOperaciones ({len(ops)} en total):")
    imprimir_operaciones(ops)

    # Verificación con enfoque optimizado
    d2 = distancia_edicion_optima(s1, s2)
    print(f"\nVerificación (espacio O(n)): {d2}  {'✓' if d2 == distancia else '✗'}")


# ====================================================================== #
#  Punto de entrada                                                       #
# ====================================================================== #

if __name__ == "__main__":
    ejemplos = [
        ("caballo", "camello"),
        ("kitten",  "sitting"),
        ("",        "hola"),
        ("algoritmo", "logaritmo"),
        ("perro", "perro") # Added an example for no changes
    ]

    for s1, s2 in ejemplos:
        mostrar_resultado(s1, s2)
        print()



  s1 = "caballo"
  s2 = "camello"

Tabla DP  (dp[i][j] = ediciones para s1[:i] → s2[:j])

           ε   c   a   m   e   l   l   o
   ε    0   1   2   3   4   5   6   7
   c    1   0   1   2   3   4   5   6
   a    2   1   0   1   2   3   4   5
   b    3   2   1   1   2   3   4   5
   a    4   3   2   2   2   3   4   5
   l    5   4   3   3   3   2   3   4
   l    6   5   4   4   4   3   2   3
   o    7   6   5   5   5   4   3   2

Distancia de edición: 2

Operaciones (7 en total):

  #    Operación    Detalle
  ------------------------------------------------
  1    = COPIAR      c en 0
  2    = COPIAR      a en 1
  3    ↔ REEMPLAZAR  b por m en 2
  4    ↔ REEMPLAZAR  a por e en 3
  5    = COPIAR      l en 4
  6    = COPIAR      l en 5
  7    = COPIAR      o en 6

Verificación (espacio O(n)): 2  ✓


  s1 = "kitten"
  s2 = "sitting"

Tabla DP  (dp[i][j] = ediciones para s1[:i] → s2[:j])

           ε   s   i   t   t   i   n   g
   ε    0   1   2   3   4   5   6   7
   k    1   1   2  